# TD — debate nhiều họ mô hình (Society of Mind theo cài đặt của Choi et al. 2025) trên Kaggle

Xem `kaggle/README.md`. Quy trình: push code lên GitHub → notebook này clone đúng một commit → cài vLLM → chạy.

**Cài đặt notebook:** Accelerator = *GPU T4 x2*, Internet = *On*, Persistence = *Files only*.
**Secrets** (Add-ons → Secrets, đều tùy chọn): `HF_TOKEN` (tải nhanh hơn; không mô hình nào cần xin quyền); `GITHUB_TOKEN` (chỉ khi repo riêng tư).
**Phiên sau:** thêm output của phiên trước làm Input; ô thứ 3 chép kết quả cũ vào để chạy tiếp.

In [ ]:
import os, glob, shutil, subprocess, time
REPO_URL = "https://github.com/kn1827/TD.git"  # repo của bạn
COMMIT = "main"                                  # nên ghi mã commit cụ thể để chạy lại đúng code
MODE = "smoke"                                   # "smoke" ở phiên đầu, sau đó "main"
VLLM_VERSION = ""                                # để trống ở phiên đầu; ghim bản đã chạy được cho các phiên sau

T0 = time.time()
os.environ["TD_DEADLINE_TS"] = str(T0 + 11.3 * 3600)   # dừng nhận việc mới trước mốc 12 giờ

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")   # tùy chọn
except Exception:
    pass
url = REPO_URL
try:
    gh = secrets.get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@")
except Exception:
    pass                                              # repo công khai: không cần token

WORK = "/kaggle/working/TD"
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "--quiet", url, WORK], check=True)
subprocess.run(["git", "-C", WORK, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", WORK, "checkout", "--quiet", COMMIT], check=True)
os.chdir(WORK)
print("code at", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
# kết quả của phiên trước (output của phiên trước được thêm làm Input)
for d in glob.glob("/kaggle/input/**/results", recursive=True):
    if not d.startswith(WORK):
        shutil.copytree(d, "/kaggle/working/results", dirs_exist_ok=True); print("restored", d); break

In [ ]:
spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
!pip install -q {spec} 2>&1 | tail -n 2
!pip install -q -r requirements.txt 2>&1 | tail -n 2
!python -c "import vllm, torch, sys; print('python', sys.version.split()[0], '| vllm', vllm.__version__, '| GPUs', torch.cuda.device_count())"
!df -h /tmp /kaggle/working | cat

In [ ]:
# kiểm thử trên CPU (~1 phút): khớp code gốc của Choi et al., chạy giả trọn quy trình
!git clone --quiet https://github.com/deeplearning-wisc/debate-or-vote.git debate-or-vote 2>/dev/null; git -C debate-or-vote checkout --quiet 82c929e
!python -m tests.test_choi_fidelity && python -m tests.test_pipeline

In [ ]:
p = subprocess.run(["bash", "kaggle/kaggle_run.sh", MODE])
print("exit code", p.returncode, "(3 = dừng trước giới hạn phiên, chạy lại ở phiên sau)")

In [ ]:
run = "smoke1" if MODE == "smoke" else "hetero1"
print(open(f"/kaggle/working/results/{run}/summary.md").read() if os.path.exists(f"/kaggle/working/results/{run}/summary.md") else "chưa có summary (run chưa xong)")
!du -sh /kaggle/working/results